In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Kirti_Singh_Resume (1).pdf to Kirti_Singh_Resume (1).pdf


In [ ]:
# Scenario: AI Research Assistant for a Corporate Innovation Team
# Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
# trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions
# about the papers instead of reading them cover to cover.

# ==========================================================
# SIMPLE RAG CHATBOT (NO LANGCHAIN) — FULLY ANNOTATED
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------

!pip install -q chromadb sentence-transformers pypdf transformers accelerate


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Load the PDF Document
# ----------------------------------------------------------

print("Loading PDF document...")

reader = PdfReader("Kirti_Singh_Resume (1).pdf")

text = ""

# Extract text from every page
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:  # prevents NoneType error
        text += page_text

print("Document Loaded")
print("Total Characters:", len(text))

print("\nPreview:\n")
print(text[:500])

# Add a check to ensure text was extracted
if not text:
    raise ValueError("No text could be extracted from the PDF. Please ensure the PDF is not empty and contains extractable text.")


# ----------------------------------------------------------
# STEP 3 — Chunk the Document
# ----------------------------------------------------------

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))
print("\nExample Chunk:\n")

# Add a check to ensure chunks are created before accessing chunks[0]
if chunks:
    print(chunks[0])
else:
    print("No chunks were created. The extracted text might be too short.")


# ----------------------------------------------------------
# STEP 4 — Create Embeddings
# ----------------------------------------------------------

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

# Delete collection if exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass

collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]

# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

print("LLM loaded successfully")


def answer_question(query):

    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
Answer the question using ONLY the context.

Context:
{context}

Question: {query}
Answer:
"""

    response = qa_pipeline(prompt)

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Loading PDF document...
Document Loaded
Total Characters: 2490

Preview:

KIRTI SINGH
Delhi | kirtisingh.dolphin@gmail.com | 8840688944 | Leetcode | Linkedin | github
Professional Summary
Computer Science Engineering student with a solid foundation in programming, data structures, and front-end
development. Proficient in HTML, CSS, JavaScript, and Java, with strong problem-solving and analytical abilities.
Quick learner and adaptable team player, passionate about creating user-friendly web applications and
contributing to innovation and organizational growth.
Educatio

Splitting document into chunks...
Total Chunks Created: 6

Example Chunk:

KIRTI SINGH
Delhi | kirtisingh.dolphin@gmail.com | 8840688944 | Leetcode | Linkedin | github
Professional Summary
Computer Science Engineering student with a solid foundation in programming, data structures, and front-end
development. Proficient in HTML, CSS, JavaScript, and Java, with strong problem-solving and analytical abilities.
Quick learner

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Old collection deleted
New vector collection created

Creating embeddings and storing in ChromaDB...
All chunks stored successfully

Loading LLM...


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

LLM loaded successfully

RAG Chatbot Ready
Type 'exit' to stop

Ask a question: what is dsa


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
Answer the question using ONLY the context.

Context:
 to innovation and organizational growth.
Education
B.Tech in Computer Science and Engineering, Galgotias University Oct 2022 – Present
• CGPA: 8.58/10.0
• Courses Operating System , Data Structure and Algorithms , Database Management System , OOPS
Intermediate(ISC), Guru Har Rai Academy 2019-2020
High School(ICSE), Guru Har Rai Academy 2017-2018
Technical Skills
Programming Languages: Java, SQL
Database: MySQL, RDBMS concepts (Normalization, Joins, Indexing)
Web Technologies: HTML, CSS, JavaScr e Solution Challenge, building impactful solutions aligned with UN Sustainable
Development Goals.
• Received Certificate of Participation in Google Summer of Code (GSoC), gaining exposure to open-source
development and global collaboration.  and optimized result display .
• Tech Stack:Flask, HTML/CSS/JavaScript, pdfplumber, SQL, Trie Data Structure.
Online Movie Ticket Booking system (June 2025-July 2025) Link
• Developed an autom

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ai_research.pdf to ai_research.pdf


In [ ]:
# Scenario: AI Research Assistant for a Corporate Innovation Team
# Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
# trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions
# about the papers instead of reading them cover to cover.

# ==========================================================
# SIMPLE RAG CHATBOT (NO LANGCHAIN) — FULLY ANNOTATED
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------

!pip install -q chromadb sentence-transformers pypdf transformers accelerate


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Load the PDF Document
# ----------------------------------------------------------

print("Loading PDF document...")

reader = PdfReader("ai_research.pdf")

text = ""

# Extract text from every page
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:  # prevents NoneType error
        text += page_text

print("Document Loaded")
print("Total Characters:", len(text))

print("\nPreview:\n")
print(text[:500])

# Add a check to ensure text was extracted
if not text:
    raise ValueError("No text could be extracted from the PDF. Please ensure the PDF is not empty and contains extractable text.")


# ----------------------------------------------------------
# STEP 3 — Chunk the Document
# ----------------------------------------------------------

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))
print("\nExample Chunk:\n")

# Add a check to ensure chunks are created before accessing chunks[0]
if chunks:
    print(chunks[0])
else:
    print("No chunks were created. The extracted text might be too short.")


# ----------------------------------------------------------
# STEP 4 — Create Embeddings
# ----------------------------------------------------------

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

# Delete collection if exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass

collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]

# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

print("LLM loaded successfully")


def answer_question(query):

    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
Answer the question using ONLY the context.

Context:
{context}

Question: {query}
Answer:
"""

    response = qa_pipeline(prompt)

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Loading PDF document...
Document Loaded
Total Characters: 2812

Preview:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and conversational agents.
Machine Learning
Machine Learning is a sub

Splitting document into chunks...
Total Chunks Created: 7

Example Chunk:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applicat

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Old collection deleted
New vector collection created

Creating embeddings and storing in ChromaDB...
All chunks stored successfully

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

...

LLM loaded successfully

RAG Chatbot Ready
Type 'exit' to stop

Ask a question: what is machine learning


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
Answer the question using ONLY the context.

Context:
agents.
Machine Learning
Machine Learning is a subset of artificial intelligence that focuses on algorithms that improve
automatically through experience. Instead of being explicitly programmed, machine learning models
learn patterns from data. Common types of machine learning include supervised learning,
unsupervised learning, and reinforcement learning.
Deep Learning
Deep learning is a specialized branch of machine learning that uses neural networks with multiple
layers to model complex patter Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
Answer the question using ONLY the context.

Context:
Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and conversational agents.
Machine Learning
Machine Learning is a sub agents.
Machine Learning
Machine Learning is a subset of artificial intelligence that focuses on algorithms that improve
automatically through experience. Instead of being explicitly programmed, machine learning models
learn patterns from data. Common types of machine learning include supervised learning,
unsupervised learning, and reinforcement learning.
Deep Learning
Deep learning is a specialized branch of machine learning that

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving data_privacy_regulation.pdf to data_privacy_regulation.pdf


In [ ]:
# Scenario: Legal Research Assistant for a Corporate Compliance Team
# Context
# A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates. These documents are dense, full of
# legal terminology, and often hundreds of pages long. The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
# - Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections) so no detail is overlooked.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, enabling semantic search rather than keyword-only lookup.
# - Retriever: When someone asks, “What does this regulation say about cross-border data transfers?”, the retriever surfaces the most relevant clauses.
# - LLM Response: A Hugging Face model (e.g., Flan-T5) generates a concise, plain-language summary of those clauses, stripping away heavy legal jargon.
# - Chat Loop: The compliance team can continue asking questions interactively, like “Does this regulation conflict with GDPR?” or “What penalties are mentioned
#  for non-compliance?”.
# Outcome
# The chatbot acts as a legal research assistant, helping the compliance team quickly interpret complex documents, identify risks, and prepare summaries for executives
#  without needing to manually parse every page.

# ==========================================================
# LEGAL RESEARCH ASSISTANT (RAG CHATBOT)
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------

!pip install -q chromadb sentence-transformers pypdf transformers accelerate


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Upload Legal PDF
# ----------------------------------------------------------

from google.colab import files

print("Upload the legal document (PDF)")
uploaded = files.upload()


# Get uploaded filename
pdf_file = list(uploaded.keys())[0]


# ----------------------------------------------------------
# STEP 3 — Load the PDF Document
# ----------------------------------------------------------

print("\nLoading legal document...")

reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print("Document Loaded Successfully")
print("Total Characters:", len(text))

print("\nPreview:\n")
print(text[:500])


if not text:
    raise ValueError("No text extracted from PDF")


# ----------------------------------------------------------
# STEP 4 — Chunk the Document
# ----------------------------------------------------------

print("\nSplitting document into sections...")

def chunk_text(text, chunk_size=800, overlap=100):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Sections Created:", len(chunks))

print("\nExample Section:\n")
print(chunks[0])


# ----------------------------------------------------------
# STEP 5 — Load Embedding Model
# ----------------------------------------------------------

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 6 — Create Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")

print("Vector database ready")


# ----------------------------------------------------------
# STEP 7 — Store Chunks in ChromaDB
# ----------------------------------------------------------

print("\nCreating embeddings and storing sections...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All sections stored successfully")


# ----------------------------------------------------------
# STEP 8 — Retriever Function
# ----------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 9 — Load LLM
# ----------------------------------------------------------

print("\nLoading language model...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

print("LLM Loaded Successfully")


# ----------------------------------------------------------
# STEP 10 — Question Answering Function
# ----------------------------------------------------------

def answer_question(query):

    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are a legal research assistant.

Use ONLY the context below to answer the question in simple language.

If the answer is not present, say:
"Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(prompt)

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 11 — Chat Loop
# ----------------------------------------------------------

print("\n=================================")
print("LEGAL RESEARCH ASSISTANT READY")
print("Type 'exit' to stop")
print("=================================\n")

while True:

    question = input("Ask a legal question: ")

    if question.lower() == "exit":
        print("Session ended.")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Upload the legal document (PDF)


Saving data_privacy_regulation.pdf to data_privacy_regulation (2).pdf

Loading legal document...
Document Loaded Successfully
Total Characters: 2885

Preview:

Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation outlines rules related to data collection, storage, processing, cross-border transfers, and
penalties for non■compliance.
Article 1: Definitions

Personal Data: Any information relating to an identified or identifiable individual.

Data Contr

Splitting document into sections...
Total Sections Created: 5

Example Section:

Data Privacy and Protection Regulation (Sample
 Document)
This sample regulation is created for educational and demonstration purposes. It simulates a legal
document that can be used for testing Retrieval-Augmented Generation (RAG) systems. The
regulation o

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Vector database ready

Creating embeddings and storing sections...
All sections stored successfully

Loading language model...


...


Answer:
 
You are a legal research assistant.

Use ONLY the context below to answer the question in simple language.

If the answer is not present, say:
"Not found in document".

Context:
 authorities within 72 hours.

Affected individuals must be informed if the breach poses a high risk to their rights and
freedoms.
Article 6: Penalties for Non■Compliance
Organizations found to be in violation of this regulation may face financial penalties, regulatory
investigations, and operational restrictions. Administrative fines may reach up to 4 percent of the
company's annual global revenue or a fixed penalty determined by the regulatory authority.
Repeated violations may result in suspension of data processing activities.
Article 7: Compliance and Audits
Organizations must maintain documentation demonstrating compliance with this regulation.
Regulatory authorities may conduct periodic audits to ensure adherence to data protection
principles and security standards.
 Data Privacy and Protecti

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are a legal research assistant.

Use ONLY the context below to answer the question in simple language.

If the answer is not present, say:
"Not found in document".

Context:
on of personal data under certain conditions.

Individuals may object to automated decision■making processes involving their data.
Article 4: Cross■Border Data Transfers
Transfer of personal data to another country or jurisdiction is permitted only when the receiving
country ensures an adequate level of data protection. Organizations must implement safeguards
such as standard contractual clauses, binding corporate rules, or regulatory approvals before
transferring personal data internationally.
Article 5: Data Security and Breach Notification

Organizations must implement encryption, access control, and monitoring systems.

Any data breach that risks individual privacy must be reported to authorities within 72 hours.

Affected individuals must be informed if the breach poses a high ris Data Priva

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are a legal research assistant.

Use ONLY the context below to answer the question in simple language.

If the answer is not present, say:
"Not found in document".

Context:
 authorities within 72 hours.

Affected individuals must be informed if the breach poses a high risk to their rights and
freedoms.
Article 6: Penalties for Non■Compliance
Organizations found to be in violation of this regulation may face financial penalties, regulatory
investigations, and operational restrictions. Administrative fines may reach up to 4 percent of the
company's annual global revenue or a fixed penalty determined by the regulatory authority.
Repeated violations may result in suspension of data processing activities.
Article 7: Compliance and Audits
Organizations must maintain documentation demonstrating compliance with this regulation.
Regulatory authorities may conduct periodic audits to ensure adherence to data protection
principles and security standards.
 Data Privacy and Protecti

In [ ]:
# ==========================================================
# UNIVERSITY LIBRARY ASSISTANT (RAG CHATBOT)
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Libraries
# ----------------------------------------------------------

!pip install -q chromadb sentence-transformers pypdf transformers accelerate


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Upload Textbook PDF
# ----------------------------------------------------------

from google.colab import files

print("Upload your textbook PDF (example: Introduction_to_Data_Science.pdf)")
uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]


# ----------------------------------------------------------
# STEP 3 — Load PDF
# ----------------------------------------------------------

print("\nLoading textbook...")

reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print("Textbook loaded successfully")
print("Total characters:", len(text))

print("\nPreview:\n")
print(text[:500])

if len(text) == 0:
    raise ValueError("No text extracted from PDF")


# ----------------------------------------------------------
# STEP 4 — Chunk the Textbook
# ----------------------------------------------------------

print("\nSplitting textbook into sections...")

def chunk_text(text, chunk_size=800, overlap=100):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total sections created:", len(chunks))
print("\nExample section:\n", chunks[0])


# ----------------------------------------------------------
# STEP 5 — Load Embedding Model
# ----------------------------------------------------------

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")


# ----------------------------------------------------------
# STEP 6 — Create Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("library_docs")
except:
    pass

collection = client.create_collection("library_docs")

print("Vector database created")


# ----------------------------------------------------------
# STEP 7 — Store Sections in ChromaDB
# ----------------------------------------------------------

print("\nCreating embeddings and storing sections...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All sections stored successfully")


# ----------------------------------------------------------
# STEP 8 — Retriever
# ----------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 9 — Load LLM
# ----------------------------------------------------------

print("\nLoading language model...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

print("Language model loaded successfully")


# ----------------------------------------------------------
# STEP 10 — Question Answering Function
# ----------------------------------------------------------

def answer_question(query):

    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are a helpful university study assistant.

Use ONLY the context below to answer the question in simple student-friendly language.

If the answer is not present, say:
"Not found in textbook".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(prompt)

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 11 — Chat Loop
# ----------------------------------------------------------

print("\n=================================")
print("UNIVERSITY LIBRARY ASSISTANT READY")
print("Type 'exit' to stop")
print("=================================\n")

while True:

    question = input("Ask your study question: ")

    if question.lower() == "exit":
        print("Session ended.")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Upload your textbook PDF (example: Introduction_to_Data_Science.pdf)


Saving Introduction_to_Data_Science.pdf to Introduction_to_Data_Science.pdf

Loading textbook...
Textbook loaded successfully
Total characters: 3427

Preview:

Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms. Data scientists analyze this data to help businesses make better decisions.
1. What is Data Science?
Data Science involves collecting, cleaning, analyzing, 

Splitting textbook into sections...
Total sections created: 5

Example section:
 Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully
Vector database created

Creating embeddings and storing sections...
All sections stored successfully

Loading language model...


...


Answer:
 
You are a helpful university study assistant.

Use ONLY the context below to answer the question in simple student-friendly language.

If the answer is not present, say:
"Not found in textbook".

Context:
 is one of the most important components of data science. It allows computers to
automatically learn patterns from data and make predictions without being explicitly programmed.
Types of Machine Learning:

Supervised Learning – Models are trained using labeled data where the correct output is
known. Examples include predicting house prices or classifying emails as spam or not spam.

Unsupervised Learning – Models analyze unlabeled data and try to find hidden patterns or
groupings. Examples include customer segmentation and clustering.

Reinforcement Learning – Algorithms learn through trial and error by receiving rewards or
penalties.
Supervised vs Unsupervised Learning
Supervised learning requires labeled datasets, meaning the algorithm is trained with both inputs
and e

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are a helpful university study assistant.

Use ONLY the context below to answer the question in simple student-friendly language.

If the answer is not present, say:
"Not found in textbook".

Context:
tion – Gathering raw data from different sources.

Data Cleaning – Removing noise, errors, or missing values.

Data Analysis – Applying statistical and computational methods.

Machine Learning – Building models that learn patterns from data.

Data Visualization – Presenting insights using charts and dashboards.
2. The Data Science Workflow
1
Define the problem or research question.
2
Collect relevant data from databases, APIs, or files.
3
Clean and preprocess the data.
4
Explore the data using statistics and visualization.
5
Build predictive or analytical models.
6
Evaluate and interpret results.
7
Communicate insights to stakeholders.
3. Machine Learning in Data Science
Machine learning is one of the most important components of data science. It allows computers to
aut

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are a helpful university study assistant.

Use ONLY the context below to answer the question in simple student-friendly language.

If the answer is not present, say:
"Not found in textbook".

Context:
Science

Healthcare – Disease prediction and medical imaging.

Finance – Fraud detection and risk analysis.

E-commerce – Recommendation systems.

Transportation – Traffic prediction and route optimization.

Social Media – Sentiment analysis and trend detection.
Conclusion
Data Science is transforming industries by enabling organizations to make data-driven decisions.
With the rapid growth of data, the demand for skilled data scientists continues to rise.
Understanding the basics of data science, machine learning, and data analysis is an essential skill
for students and professionals in the modern digital economy.
 tion – Gathering raw data from different sources.

Data Cleaning – Removing noise, errors, or missing values.

Data Analysis – Applying statistical and co

In [ ]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# ==============================================================

# -------------------------------
# STEP 1 — Install Dependencies
# -------------------------------

!pip install -q gradio chromadb sentence-transformers pypdf transformers accelerate torch


# -------------------------------
# STEP 2 — Imports
# -------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# -------------------------------
# STEP 3 — Load Embedding Model
# -------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# -------------------------------
# STEP 4 — Initialize Vector DB
# -------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")


# -------------------------------
# STEP 5 — Load LLM
# -------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=200
)

print("LLM loaded successfully")


# -------------------------------
# STEP 6 — Chunk Function
# -------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


# -------------------------------
# STEP 7 — Process PDF
# -------------------------------

def process_pdf(file):

    if file is None:
        return "Please upload a PDF."

    print("Processing PDF...")

    reader = PdfReader(file)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text

    if text.strip() == "":
        return "No readable text found in PDF."

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# -------------------------------
# STEP 8 — Retrieve Documents
# -------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    return docs


# -------------------------------
# STEP 9 — Generate Answer
# -------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
Use the context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm(prompt)

    return response[0]["generated_text"]


# -------------------------------
# STEP 10 — Chat Function
# -------------------------------

def chat(question):

    if question is None or question.strip() == "":
        return "Please enter a question."

    return answer_question(question)


# -------------------------------
# STEP 11 — Gradio Interface
# -------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a regulation PDF and ask questions about:

• compliance rules
• penalties
• privacy laws
""")

    pdf_input = gr.File(file_types=[".pdf"])

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(label="Ask Question")

    answer_box = gr.Textbox(label="Answer", lines=10)

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# -------------------------------
# STEP 12 — Launch App
# -------------------------------

demo.launch(share=True)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

LLM loaded successfully
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30e721cc0211b5d580.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ==============================================================
# HEALTHCARE POLICY NAVIGATOR
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

!pip install gradio chromadb sentence-transformers pypdf transformers

# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("healthcare_docs")
except:
    pass

collection = client.create_collection("healthcare_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing healthcare regulation document...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Healthcare regulation processed. {len(chunks)} sections stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Sections:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a Healthcare Policy Compliance Assistant.

Your job is to help doctors, hospital administrators,
and IT teams understand healthcare regulations.

Use ONLY the context below to answer the question clearly.

Context:
{context}

Question: {query}

Provide a clear and actionable answer.
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )

    result = response[0]["generated_text"]

    print("\nModel Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a healthcare policy question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# ↓↓ Healthcare Policy Navigator")

    gr.Markdown("""
Upload a healthcare regulation document and ask questions about:

• patient data privacy
• telemedicine regulations
• compliance requirements
• penalties for violations
""")

    pdf_input = gr.File(label="Upload Healthcare Policy PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Healthcare Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b2e458010366f0ccc2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# ENVIRONMENTAL POLICY COMPLIANCE ASSISTANT (RAG SYSTEM)
# ============================================================

# ------------------------------------------------------------
# STEP 1 — Install Dependencies
# ------------------------------------------------------------

!pip install -q gradio chromadb sentence-transformers pypdf transformers accelerate torch


# ------------------------------------------------------------
# STEP 2 — Import Libraries
# ------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# ------------------------------------------------------------
# STEP 3 — Load Embedding Model
# ------------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ------------------------------------------------------------
# STEP 4 — Initialize Vector Database
# ------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("environmental_docs")
except:
    pass

collection = client.create_collection("environmental_docs")


# ------------------------------------------------------------
# STEP 5 — Load Language Model
# ------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=200
)

print("LLM loaded successfully")


# ------------------------------------------------------------
# STEP 6 — Text Chunking
# ------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


# ------------------------------------------------------------
# STEP 7 — Process Environmental Regulation PDF
# ------------------------------------------------------------

def process_pdf(file):

    if file is None:
        return "Please upload an environmental regulation PDF."

    print("Processing Environmental Regulation...")

    reader = PdfReader(file)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text

    if text.strip() == "":
        return "No readable text found in the PDF."

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Environmental regulation processed. {len(chunks)} sections stored."


# ------------------------------------------------------------
# STEP 8 — Retrieve Relevant Sections
# ------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    return docs


# ------------------------------------------------------------
# STEP 9 — Generate Compliance Answer
# ------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are an Environmental Compliance Assistant.

Use ONLY the regulation context below.

Context:
{context}

Question:
{query}

Provide a short compliance-focused answer.
"""

    response = llm(prompt)

    return response[0]["generated_text"]


# ------------------------------------------------------------
# STEP 10 — Chat Function
# ------------------------------------------------------------

def chat(question):

    if question is None or question.strip() == "":
        return "Please enter a compliance question."

    return answer_question(question)


# ------------------------------------------------------------
# STEP 11 — Gradio Interface
# ------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🌍 Environmental Policy Compliance Assistant")

    gr.Markdown("""
Upload an environmental regulation PDF and ask questions about:

• Carbon emission limits
• Waste disposal rules
• Renewable energy targets
• Environmental penalties
""")

    pdf_input = gr.File(file_types=[".pdf"])

    upload_button = gr.Button("Process Regulation Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask an Environmental Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Assistant Response",
        lines=10
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# ------------------------------------------------------------
# STEP 12 — Launch Application
# ------------------------------------------------------------

demo.launch(share=True)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM loaded successfully
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf37bc813c1f2e5d76.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
